# LoRA and Parameter-Efficient Fine-Tuning


In [1]:
from __future__ import annotations

from pathlib import Path

import torch

LECTURE_NOTE_TITLE = 'LoRA and Parameter-Efficient Fine-Tuning'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: LoRA and Parameter-Efficient Fine-Tuning


## Demo A: Why low-rank updates are cheaper


In [2]:
def parameter_counts(in_features: int, out_features: int, rank: int) -> dict[str, int]:
    dense = in_features * out_features
    lora = rank * in_features + rank * out_features
    return {"dense": dense, "lora": lora, "reduction_factor": round(dense / lora, 2)}

for rank in [4, 8, 16, 32]:
    print(rank, parameter_counts(4096, 4096, rank))


4 {'dense': 16777216, 'lora': 32768, 'reduction_factor': 512.0}
8 {'dense': 16777216, 'lora': 65536, 'reduction_factor': 256.0}
16 {'dense': 16777216, 'lora': 131072, 'reduction_factor': 128.0}
32 {'dense': 16777216, 'lora': 262144, 'reduction_factor': 64.0}


## Demo B: Approximate a dense update with a low-rank factorization


In [3]:
target_delta = torch.tensor(
    [
        [0.10, -0.06, 0.00],
        [0.04, 0.02, -0.03],
        [0.08, -0.02, 0.01],
    ],
    dtype=torch.float32,
)
u, s, vh = torch.linalg.svd(target_delta)
rank = 2
A = u[:, :rank] @ torch.diag(torch.sqrt(s[:rank]))
B = torch.diag(torch.sqrt(s[:rank])) @ vh[:rank, :]
approx = A @ B
print('target delta:\n', target_delta)
print('\napprox delta:\n', approx)
print('\nrelative error:', float(torch.norm(target_delta - approx) / torch.norm(target_delta)))


target delta:
 tensor([[ 0.1000, -0.0600,  0.0000],
        [ 0.0400,  0.0200, -0.0300],
        [ 0.0800, -0.0200,  0.0100]])

approx delta:
 tensor([[ 0.1022, -0.0547,  0.0079],
        [ 0.0410,  0.0224, -0.0265],
        [ 0.0765, -0.0284, -0.0025]])

relative error: 0.1228056401014328


## Demo C: Trainable-parameter split in a transformer-style projection


In [4]:
projection_shapes = {
    'q_proj': (4096, 4096),
    'k_proj': (4096, 4096),
    'v_proj': (4096, 4096),
    'o_proj': (4096, 4096),
}
rank = 16
summary = {}
for name, (inp, out) in projection_shapes.items():
    summary[name] = parameter_counts(inp, out, rank)
summary


{'q_proj': {'dense': 16777216, 'lora': 131072, 'reduction_factor': 128.0},
 'k_proj': {'dense': 16777216, 'lora': 131072, 'reduction_factor': 128.0},
 'v_proj': {'dense': 16777216, 'lora': 131072, 'reduction_factor': 128.0},
 'o_proj': {'dense': 16777216, 'lora': 131072, 'reduction_factor': 128.0}}

## Demo D: Real workflow mapping in this repo


In [5]:
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
for path in [
    'picollm/sft_lora/prepare_dataset.py',
    'picollm/sft_lora/finetune.py',
    'picollm/sft_lora/evaluate.py',
]:
    print(path)


picollm/sft_lora/prepare_dataset.py
picollm/sft_lora/finetune.py
picollm/sft_lora/evaluate.py


## Compare with Rasbt and nanochat


In [6]:
comparison = {
    'course repo': 'teach low-rank intuition first, then show a practical LoRA workflow',
    'rasbt': 'teaches the fine-tuning arc clearly from first principles',
    'nanochat': 'shows the product-scale training and systems path after the concept is already clear',
}
comparison


{'course repo': 'teach low-rank intuition first, then show a practical LoRA workflow',
 'rasbt': 'teaches the fine-tuning arc clearly from first principles',
 'nanochat': 'shows the product-scale training and systems path after the concept is already clear'}